# Fire Detection on Images (YOLO26 + Supervision)

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Runs the fire-detection model trained in `drone_fire_detection_yolo26.ipynb` on a single image and draws
the results with [Supervision](https://supervision.roboflow.com/).

A GPU is optional here — single-image inference is fine on CPU.

## 1. Install

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0"

import supervision as sv
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("supervision:", sv.__version__)

## 2. Get the model weights and a test image

`best.pt` is not committed to the repo — upload the checkpoint you exported from
`drone_fire_detection_yolo26.ipynb`. The sample image is pulled from the repo automatically.

In [ ]:
from pathlib import Path

MODEL_PATH = Path("best.pt")
IMAGE_PATH = Path("fire_image.png")

if not IMAGE_PATH.exists():
    !wget -q -O {IMAGE_PATH} https://raw.githubusercontent.com/jakkzz/Fire-Detection-Drone/main/fire_image.png

if not MODEL_PATH.exists():
    print("best.pt not found — upload it now (or run the training notebook first).")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("Place best.pt next to this notebook.")

print("model:", MODEL_PATH.resolve())
print("image:", IMAGE_PATH.resolve())

## 3. Run inference

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f"Could not read {IMAGE_PATH}")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

print("detections:", len(detections))

## 4. Annotate

Modern Supervision splits drawing across separate annotators — `BoxAnnotator` no
longer accepts a `labels=` argument (removed in supervision 0.22), so labels are
drawn by `LabelAnnotator`.

Class names come from `detections["class_name"]`, which
`Detections.from_ultralytics` fills in from the model itself, so there is no
hand-maintained class list to drift out of sync.

In [ ]:
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8, text_thickness=2, text_padding=6)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

# OpenCV loads BGR; convert so the colours are right in matplotlib.
sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))

## 5. Save the annotated image

In [ ]:
OUTPUT_PATH = Path("fire_image_annotated.png")
cv2.imwrite(str(OUTPUT_PATH), annotated)
print("saved:", OUTPUT_PATH.resolve())